# 07 — Cross-Dataset Validation

**Purpose:** Evaluate dataset shift and assess generalisability of signal quality
and morphology features across `ptbxl`, `ludb`, and `nstdb`.

**Output:** `outputs/cross_dataset_results.parquet`

**Granularity:** Dataset level — primary key: `dataset_name`

**Data Contract:** `DATA_CONTRACT.md` §14


In [1]:
import sys
sys.path.insert(0, '../src')


In [2]:
import numpy as np
import pandas as pd
from datetime import datetime
from scipy.stats import ks_2samp

RANDOM_SEED = 42
PIPELINE_VERSION = "1.0.0"
rng = np.random.default_rng(RANDOM_SEED)
TIMESTAMP = datetime.utcnow().isoformat()

sq_df    = pd.read_parquet("../outputs/signal_quality_features.parquet")
twave_df = pd.read_parquet("../outputs/twave_features.parquet")
inventory = pd.read_csv("../outputs/inventory.csv")

sq_merged = sq_df.merge(inventory[["record_id","dataset_name"]], on="record_id")
tw_merged = twave_df.merge(inventory[["record_id","dataset_name"]], on="record_id")
print("Signal quality shape  :", sq_merged.shape)
print("T-wave features shape :", tw_merged.shape)


Signal quality shape  : (740, 14)
T-wave features shape : (3700, 15)


/tmp/ipykernel_2838/3769336279.py:9: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  TIMESTAMP = datetime.utcnow().isoformat()


## Distribution Shift Analysis

We use the Kolmogorov-Smirnov statistic to quantify distributional shift between
each dataset pair across key features.


In [3]:
DATASETS = sq_merged["dataset_name"].unique().tolist()
SQ_FEATURES = ["snr_db","bw_index","hfn_index","pli_index","signal_quality_score"]
TW_FEATURES = ["t_end_ambiguity_score","morphology_confidence","t_amplitude_mv"]

def ks_shift(df, col, ds_a, ds_b):
    a = df[df.dataset_name == ds_a][col].dropna().values
    b = df[df.dataset_name == ds_b][col].dropna().values
    if len(a) < 5 or len(b) < 5:
        return np.nan
    stat, _ = ks_2samp(a, b)
    return float(stat)

rows = []
for ds in DATASETS:
    others = [d for d in DATASETS if d != ds]
    sq_shifts = []
    tw_shifts = []
    conf_stab = []

    for other in others:
        for feat in SQ_FEATURES:
            sq_shifts.append(ks_shift(sq_merged, feat, ds, other))
        for feat in TW_FEATURES:
            tw_shifts.append(ks_shift(tw_merged, feat, ds, other))

    shift_score               = float(np.nanmean(sq_shifts + tw_shifts))
    lead_distribution_shift   = float(np.nanmean(sq_shifts))
    morphology_shift          = float(np.nanmean(tw_shifts))
    # Confidence stability: lower shift → more stable
    confidence_stability_score = float(np.clip(1.0 - shift_score, 0, 1))

    rows.append({
        "dataset_name":              ds,
        "shift_score":               shift_score,
        "lead_distribution_shift":   lead_distribution_shift,
        "morphology_shift":          morphology_shift,
        "confidence_stability_score": confidence_stability_score,
        "pipeline_version": PIPELINE_VERSION,
        "processing_timestamp": TIMESTAMP,
    })

cross_df = pd.DataFrame(rows)
print(cross_df[["dataset_name","shift_score","confidence_stability_score"]].to_string(index=False))


dataset_name  shift_score  confidence_stability_score
       ptbxl     0.169766                    0.830234
        ludb     0.177161                    0.822839
       nstdb     0.271771                    0.728229


## Schema Validation

In [4]:
REQUIRED_CROSS_COLS = [
    "dataset_name","shift_score","lead_distribution_shift",
    "morphology_shift","confidence_stability_score",
]
missing = [c for c in REQUIRED_CROSS_COLS if c not in cross_df.columns]
assert not missing, f"Missing: {missing}"
assert cross_df["confidence_stability_score"].between(0,1).all()
print("✓ Schema validation passed")
print(cross_df.to_string(index=False))


✓ Schema validation passed
dataset_name  shift_score  lead_distribution_shift  morphology_shift  confidence_stability_score pipeline_version       processing_timestamp
       ptbxl     0.169766                 0.232292          0.065556                    0.830234            1.0.0 2026-06-13T05:06:01.705576
        ludb     0.177161                 0.243333          0.066875                    0.822839            1.0.0 2026-06-13T05:06:01.705576
       nstdb     0.271771                 0.368958          0.109792                    0.728229            1.0.0 2026-06-13T05:06:01.705576


## Visualisation

In [5]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
cross_df.set_index("dataset_name")[["lead_distribution_shift","morphology_shift"]].plot.bar(
    ax=axes[0], color=["#1f77b4","#ff7f0e"], edgecolor="white")
axes[0].set_title("Feature Distribution Shift (KS statistic)")
axes[0].set_ylabel("KS statistic")
axes[0].tick_params(axis='x', rotation=0)

cross_df.set_index("dataset_name")["confidence_stability_score"].plot.bar(
    ax=axes[1], color="#2ca02c", edgecolor="white")
axes[1].set_title("Confidence Stability Score")
axes[1].set_ylabel("Score (0-1)")
axes[1].set_ylim(0, 1.1)
axes[1].tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig("../outputs/cross_dataset_summary.png", dpi=100)
plt.show()
print("Figure saved.")


Figure saved.


/tmp/ipykernel_2838/3302755739.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Export

In [6]:
cross_df.to_parquet("../outputs/cross_dataset_results.parquet", index=False)
print("✓ cross_dataset_results.parquet →", cross_df.shape)


✓ cross_dataset_results.parquet → (3, 7)
